In [21]:
# Cài đặt thư viện transformers và accelerate để chạy mô hình AI
!pip install -q transformers accelerate

In [23]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Khởi tạo Mô hình và Tokenizer (Chạy trên GPU)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print("Đang tải mô hình, vui lòng đợi...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
print("Tải mô hình thành công!")

# 2. Định nghĩa hàm trích xuất thông tin động (Dynamic UIE)
def dynamic_extraction_uie(raw_text, schema):
    # Chuyển đổi schema thành định dạng JSON rỗng để làm mẫu (Prompt Engineering)
    schema_template = {key: {field: "" for field in value} for key, value in schema.items()}

    # Xây dựng Prompt ép mô hình trả về đúng cấu trúc Schema (SSI)
    system_prompt = f"""Bạn là một chuyên gia trích xuất dữ liệu hành chính (Universal Information Extraction).
Nhiệm vụ của bạn là đọc văn bản OCR và trích xuất thông tin khớp chính xác với CẤU TRÚC JSON MẪU dưới đây.
- Chỉ trả về duy nhất chuỗi JSON hợp lệ, tuyệt đối không giải thích hay thêm văn bản nào khác.
- Nếu không tìm thấy thông tin trong văn bản, hãy để giá trị là null.

[CẤU TRÚC JSON MẪU BẮT BUỘC]:
{json.dumps(schema_template, ensure_ascii=False, indent=2)}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"[VĂN BẢN OCR CẦN XỬ LÝ]:\n{raw_text}"}
    ]

    # Chuẩn bị input cho mô hình
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Sinh kết quả (Inference)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature=0.1, # Đặt temperature thấp để AI ưu tiên tính chính xác, không sáng tạo
        do_sample=True
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Làm sạch kết quả trả về để đảm bảo lấy được JSON
    try:
        if "```json" in response:
            response = response.split("```json")[1].split("```")[0].strip()
        elif "```" in response:
            response = response.split("```")[1].strip()
        return json.loads(response)
    except Exception as e:
        print("Lỗi parse JSON:", e)
        return {"raw_output": response}

# 3. Đọc dữ liệu đầu vào và chạy vòng lặp xử lý
input_file = '/content/drive/MyDrive/VNDigitize/UIEInputdata.json'
output_file = '/content/drive/MyDrive/VNDigitize/UIE_Extraction_Results.json'

with open(input_file, 'r', encoding='utf-8') as f:
    documents = json.load(f)

results = []

print(f"\nBắt đầu xử lý {len(documents)} tài liệu...")
for doc in documents:
    doc_id = doc.get('document_id', 'Unknown')
    print(f"\nĐang trích xuất tài liệu: {doc_id}")

    raw_text = doc['module2_output']['raw_text']
    instruction_schema = doc['instruction_schema']

    # Chạy hàm UIE động
    extracted_data = dynamic_extraction_uie(raw_text, instruction_schema)

    # Đóng gói kết quả
    doc_result = {
        "document_id": doc_id,
        "basic_metadata": doc['basic_metadata'],
        "extracted_dynamic_data": extracted_data
    }
    results.append(doc_result)

    print(f"Hoàn thành {doc_id}! Kết quả trích xuất:")
    print(json.dumps(extracted_data, ensure_ascii=False, indent=2))

# 4. Lưu kết quả ra file JSON mới
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nĐã lưu toàn bộ kết quả trích xuất vào tệp: {output_file}")

Đang tải mô hình, vui lòng đợi...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tải mô hình thành công!

Bắt đầu xử lý 3 tài liệu...

Đang trích xuất tài liệu: TAYNINH_2025_03_HCPT
Hoàn thành TAYNINH_2025_03_HCPT! Kết quả trích xuất:
{
  "Người khởi kiện": {
    "Họ tên": "Bà Trần Thị Thu V",
    "Năm sinh": "1944",
    "Số CCCD": "072144000198",
    "Địa chỉ": "khu phố B, phường B, tỉnh Tây Ninh",
    "Trạng thái có mặt": "vắng mặt"
  },
  "Người bị kiện": {
    "Tên tổ chức/Họ tên": "Ủy ban nhân dân xã B (nay là phường B)",
    "Địa chỉ": "tỉnh Tây Ninh",
    "Người đại diện": "Ông Lương Bá C",
    "Chức vụ đại diện": "Chủ tịch",
    "Trạng thái có mặt": "vắng mặt"
  }
}

Đang trích xuất tài liệu: DANANG_2025_172_HCPT
Hoàn thành DANANG_2025_172_HCPT! Kết quả trích xuất:
{
  "Người khởi kiện": {
    "Họ tên": "Nguyễn Văn Đ",
    "Năm sinh": "1969",
    "Địa chỉ": "thôn N, xã P, huyện N, tỉnh Ninh Thuận (nay là xã P, tỉnh Khánh Hòa)",
    "Trạng thái có mặt": "có mặt"
  },
  "Người bị kiện": {
    "Tên tổ chức/Họ tên": "Chủ tịch Ủy ban nhân dân huyện N",
    "Chức